# Tutorial: Operator learning
### Fourier neural operator
Example PDE: 1D Burgers equation

---

Lecture: "Physics-augmented machine learning" @ Cyber-Physical Simulation, TU Darmstadt

Lecturer: Prof. Oliver Weeger

Author: Fabian J. Roth

Inspiration and parts of this notebook are taken from [Ceyron's GitHub](https://github.com/Ceyron/FNO-in-JAX/blob/main/README.md), who also provides a detailed [YouTube Video](https://www.youtube.com/watch?v=74uwQsBTIVo) about a FNO implementation.

---

#### In this notebook, you will...

* <span style="color:red">learn something</span>

# Installation
Run the following code cell to install the required packages. Then **restart your session** to make these packages available.

In [1]:
def is_colab():
    """Determine if the code is running in Google Colab."""
    try:
        import google.colab

        return True
    except ImportError:
        return False


if is_colab():
    print("Running in Google Colab, trying to install LecturePhysicsAwareML...")
    !git clone --depth 1 https://github.com/CPShub/LecturePhysicsAwareML.git

    # Get the data Mathworks (the creators of Matlab) host the original Li et al. dataset in the .mat format
    !wget https://ssd.mathworks.com/supportfiles/nnet/data/burgers1d/burgers_data_R10.mat

    # Install the package
    %cd LecturePhysicsAwareML/operator_learning
    %pip install -e .
    print(
        "\n************************************************************************************\nMake sure to restart the session after installation (Runtime > Restart Session).\n********************************************************************************"
    )
else:
    print(
        "Not running in Google Colab. \nPlease add the data and install the package manually if needed. The data can be downloaded here: https://ssd.mathworks.com/supportfiles/nnet/data/burgers1d/burgers_data_R10.mat. \nIf you are using pip, run:\n>>> pip install -e .\nin the operator_learning directory."
    )


Not running in Google Colab. 
Please add the data and install the package manually if needed. The data can be downloaded here: https://ssd.mathworks.com/supportfiles/nnet/data/burgers1d/burgers_data_R10.mat. 
If you are using pip, run:
>>> pip install -e .
in the operator_learning directory.


## Import required packages

In [2]:
import jax
import jax.numpy as jnp
import jax.random as jr

from operator_learning import FNO
from klax import split_data, fit

import optax

from scipy.io import loadmat

from matplotlib import pyplot as plt
from ipywidgets import interact, BoundedIntText, Dropdown, Button, Output, VBox, Checkbox

jax.config.update("jax_enable_x64", True)

key = jr.key(0)
key_split, model_key, training_key = jr.split(key, 3)

colors = { 
    'green':        '#16a48a',
    'lightblue':    '#688fc6',
    'darkblue':     '#435384',
    'grey':         '#cccccc',
    'orange':       '#f6a315',
    'red':          '#c24c4c',
    'black':        '#000000',
}

# Example PDE: 1D Burgers equation

In this notebook, we will mimic one example done in the original FNO paper by [Li et al.](https://arxiv.org/pdf/2010.08895.pdf) as implemented in their [reference code](https://github.com/neuraloperator/neuraloperator/blob/af93f781d5e013f8ba5c52baa547f2ada304ffb0/fourier_1d.py) to solve the [**1D Burgers equation**](https://en.wikipedia.org/wiki/Burgers%27_equation)

$$
\begin{align}
    \frac{\partial u}{\partial t} + \frac{1}{2}\frac{\partial u^2}{\partial x} = \nu \frac{\partial^2 u}{\partial x^2}, & \qquad x\in(0,1),\,t\in(0,1]\\
    u(0, x) = u_0(x), & \qquad x\in(0,1)
\end{align}
$$
$$  $$

The domain $\Omega = (0, 1)$ is periodic, i.e., $u(t, x=0) = u(t, x=1)$. The diffusivity is fixed to $\nu=0.1$. Our dataset consists of $2048$ initial conditions $u(t=0, x)$ and the corresponding solution at time one $u(t=1, x)$. The functions are sampled at $N=8192$ equidistant points $x_i$. 

Our goal is to fit an FNO to learn the mapping from initial condition $u(t=0, x)$ to state at time one $u(t=1, x)$. 

Let's have a look at the data.

In [3]:
# Load the data
try:
    data = loadmat(R"/content/burgers_data_R10.mat")
except FileNotFoundError:
    data = loadmat(R"../data/burgers_data_R10.mat")

u0s = data["a"]  # u(t=0, x)
u0s_smooth = data["a_smooth"]  # smoothed u(t=0, x)
u1s = data["u"]  # u(t=1, x)
xs = jnp.linspace(0, 1, u0s.shape[1])
xs_batched = jnp.repeat(xs[None, :], u0s.shape[0], axis=0)

In [4]:
@interact(
    sample_index=BoundedIntText(
        min=0, max=u0s.shape[0] - 1, step=1, value=0, description="Sample Index"
    )
)
def update_plot(sample_index):
    fig, ax = plt.subplots(figsize=(7, 3.5), dpi=150)
    ax.plot(xs, u0s[sample_index], c=colors['darkblue'], label="Initial condition $u(t=0, x)$")   # type: ignore
    ax.plot(xs, u1s[sample_index], c=colors['orange'], label="$u(t=1, x)$") # type: ignore
    ax.set(xlabel="$x$", ylabel="$u$", title=f"Data sample {sample_index}", xlim=[0, 1])
    ax.legend(loc=4)
    plt.tight_layout()

interactive(children=(BoundedIntText(value=0, description='Sample Index', max=2047), Output()), _dom_classes=(…

# Theory: Nerual operators and Fourier neural operators (FNOs)
Neural Operators are mappings between discretized function spaces, for example:

* Map from an initial condition to the solution function at a later point in time (or to the entire spatiotemporal solution funciton)
* Map from the function describing an inhomogeneous diffusivity distribution to the solution of the heat equation
* Autoregressive timesteppers, map state $u^{[t]}_h$ to state $u_h^{[t+1]}$

Fouries Neural Operators do so by employing the FFT to perform efficient **spectral convolution** taking into account global features. In that sense they are a multiscale architecture (Classical convolutional architectures are only local and their receptive field depends on the depth of the network).

Neural Operators allow for the solution of a whole parametric family of PDEs!

FNOs allow for **zero-shot superresolution**.

### Spectral Convolutions

Given the (real-valued) input discretized state $a$ (with potentially more than one channel) defined on an equidistant mesh; do the following steps:

1. Transform $a$ into Fourier space (here using the real-valued Fourier transform): $\hat{a} = \text{rfft}(a)$ (batch over the channel dimension)
2. Perform a batched matrix multiplication with a complex-valued weight vector $W$ for the first $K$ modes: $\hat{\tilde{a}}_{0:K} = W\hat{a}_{0:K}$
3. Set all the leftover modes to zero $\hat{\tilde{a}}_{K:} = 0 + 0i$
4. Transform back into real space $\tilde{a} = \text{irfft}(\hat{\tilde{a}})$

The learnable parameters for each spectral convolution are the complex-valued weight matrix of shape `(channels_out, channels_in, modes)` (Since it is complex-valued it actually has `2 * channels_out * channels_in * modes` real parameters)

### Fourier Neural Operator

A classical FNO consists of a lifting layer, multiple "ResNet"-like blocks of spectral convolutions with a bypass, and a projection layer. Projection and Lifting layer are local operators and only modify the channel dimensions. 

# Task A: Fitting an FNO to data from Burgers equation

Run the following code cells to _prepare the data_, _train_ and _evaluate_ an FNO.

Questions to think about:
1) What influence do the different parameters have on the training process and the FNO performance?
2) Why might the example PDE here be an easy problem for an FNO?

# Data subsampling and enrichment

#### Subsampling
Learning the map between the original data is computationally expensive, due to the fine resolution of the sampling ($N=8192$). Let's subsample the functions by keeping only every n-th element in the arrays. Because $8192 = 2^{13}$ we can easily subsample by repeatedly halfing the array.
We will call the number of times we half the "_downsampling level_". 

#### Enrichment
When training neural operators it can be helpful to add additional information to the input. For example we could use the array of $x$-values in addition to the initial condition $u(t=0,x)$. We could also add a smoothed out version of $u(t=0,x)$. The hope is that using the information in these additional inputs makes learning the operator easier. We add the additional input data as channels to our input array. If we use $u(t=0,x),\,x$ and a smoothed $u(t=0,x)$ then our input to the FNO will be of `shape=(N, 3)` where N is the number of spacial samples and 3 is our channel dimension. The FFT will only be computed along the spacial dimension, but the complex weights in the FNO layers will be able to mix the information in the channels.

In [10]:
# Pick training data parameters
print("Pick your training data parameters")
@interact(
    in_data_option=Dropdown(options=["Only u0", "u0 and x", "u0, x and u0_smooth"], value="Only u0", description="Enrichment: Input data = ", style={"description_width": "initial"}),
    se=Dropdown(options=[5, 6, 7, 8, 9], value=9, description="Downsampling level:", style={"description_width": "initial"}),
    half_data = Checkbox(value=False, description='Half domain (Only in task B!)', disabled=False, indent=False, style={"description_width": "initial"})
)
def make_input_data(in_data_option, se, half_data):
    global dataset_fine, dataset_coarse, x_coarse, x_fine, xlim
    if in_data_option == "Only u0":
        in_data = u0s[:, :, None]  # Add channel dimension
    elif in_data_option == "u0 and x":
        in_data = jnp.stack([u0s, xs_batched], axis=-1)  # Add x channel
    elif in_data_option == "u0, x and u0_smooth":
        in_data = jnp.stack([u0s, xs_batched, u0s_smooth], axis=-1)  # Add x channel
    else:
        raise ValueError("Unkown in_data_option")


    stride = 2**se
    if half_data:
        dataset_coarse = (in_data[:, :4096:stride], u1s[:, :4096:stride])
        x_coarse = xs[:4096:stride]
        dataset_fine = (in_data[:,:4096], u1s[:,:4096])
        x_fine = xs[:4096]
        xlim = [0, 0.5]
    else:
        dataset_coarse = (in_data[:, ::stride], u1s[:, ::stride])
        x_coarse = xs[::stride]
        dataset_fine = (in_data, u1s)
        x_fine = xs
        xlim = [0, 1]

    fig, ax = plt.subplots(figsize=(7, 3.5), dpi=150)
    for data, label, c in zip(
        dataset_coarse[0][0].T,
        ["$u(x,t=0)$", "x", "smoothed $u(x,t=0)$"],
        ["darkblue", "grey", "lightblue"],
    ):
        ax.plot(x_coarse, data, marker="o", c=colors[c], label=label)
    ax.plot(x_coarse, dataset_coarse[1][0], marker="s", c="orange", label="Solution $u(x,t=1)$")
    ax.set(xlabel="$x$", ylabel="$u$", title="Downsampled data example", xlim=xlim)
    ax.legend()
    plt.tight_layout()
    plt.show()



Pick your training data parameters


interactive(children=(Dropdown(description='Enrichment: Input data = ', options=('Only u0', 'u0 and x', 'u0, x…

# Train the FNO

With the following cell you can choose some parameters and train the FNO. 

| Name                       | Variable name     | Description                                                                 |
|----------------------------|-------------------|-----------------------------------------------------------------------------|
| Number of Fourier modes    | num_modes         | Controls frequency resolution in FNO layers ($\omega_{max}$)                |
| Number of hidden channels  | hidden_channels   | Width of each FNO layer                                                     |
| Number of FNO layers       | depth             | Number of FNO layers (network depth)                                        |
| Number of training steps   | steps             | Number of training steps (iterations for optimizer)                         |

In [7]:
num_modes = BoundedIntText(min=1, max=64, step=1, value=9, description="Number of Fourier modes: ", style={"description_width": "initial"})
hidden_channels = BoundedIntText(min=1, max=32, step=1, value=8, description="Number of hidden channels: ", style={"description_width": "initial"})
depth = BoundedIntText(min=1, max=8, step=1, value=2, description="Number of FNO layers: ", style={"description_width": "initial"})
steps = Dropdown(options=[1000, 5000, 10_000, 20_000], value=10_000, description="Number of training steps: ", style={"description_width": "initial"})
train_button = Button(description="Train FNO", button_style="success")
output = Output()

def train_fno_callback(b):
    with output:
        output.clear_output()
        global fno, train_data_coarse, test_data_coarse, train_data_fine, test_data_fine

        # Split the data into test and training sets
        train_data_coarse, test_data_coarse = split_data(dataset_coarse, (0.9, 0.1), key=key_split)
        train_data_fine, test_data_fine = split_data(dataset_fine, (0.9, 0.1), key=key_split)   # Same split for the fine dataset
        in_channels = train_data_coarse[0].shape[-1]

        fno = FNO(
            in_channels, "scalar", num_modes=num_modes.value, hidden_channels=hidden_channels.value, depth=depth.value, key=model_key
        )
        try:
            fno, history = fit(
                fno,
                train_data_coarse,
                batch_size=64,
                validation_data=test_data_coarse,
                steps=steps.value,
                optimizer=optax.adam(1e-4),
                key=training_key,
            )
        except ValueError as e:
            print(e)
            return

        history.plot()  # type: ignore
        plt.tight_layout()
        plt.show()
        
train_button.on_click(train_fno_callback)
VBox([num_modes, hidden_channels, depth, steps, train_button, output])

# Model evaluation

Let's evaluate the FNO on some test data

In [8]:
# Model evaluation
u0_test, u1_test = test_data_coarse

@interact(
    sample_index=BoundedIntText(
        min=0, max=u0_test.shape[0] - 1, step=1, value=0, description="Sample Index"
    )
)
def prediction_plot(sample_index):

    model_input = u0_test[sample_index]
    model_output = fno(model_input)         # Evaluate the model

    fig, ax = plt.subplots(figsize=(7, 3.5), dpi=150)
    ax.plot(x_coarse, model_input[:, 0], marker="o", c="black", label="Initial condition $u(t=0, x)$")
    ax.plot(x_coarse, u1_test[sample_index], marker="s", c=colors["orange"], label="True $u(t=1, x)$")
    ax.plot(x_coarse, model_output, c=colors["red"], ls="--", marker="s", label="Predicted $u(t=0, x)$")
    ax.set(xlabel="$x$", ylabel="$u$", title=f"Test data sample {sample_index}", xlim=xlim)
    ax.legend(loc=4)
    plt.tight_layout()

interactive(children=(BoundedIntText(value=0, description='Sample Index', max=204), Output()), _dom_classes=('…

# Zero-shot super-resolution

Let's test the claim of zero-shot super-resolution by evaluating the FNO on the original data with the very fine discretization

In [9]:
## Try 0-shot superresolution

@interact(
    sample_index=BoundedIntText(
        min=0, max=u0_test.shape[0] - 1, step=1, value=0, description="Sample Index"
    )
)
def superresolution_plot(sample_index):

    u0_test_coarse, u1_test_coarse = test_data_coarse
    u0_test_fine, u1_test_fine = test_data_fine

    model_output_coarse = fno(u0_test_coarse[sample_index])
    model_output_fine   = fno(u0_test_fine[sample_index])

    fig, axes = plt.subplots(1, 2, figsize=(9, 4), dpi=150, sharey=True)
    axes[0].plot(x_coarse, u0_test_coarse[sample_index, :, 0], marker="o", c="black", label="Initial condition $u(t=0, x)$")
    axes[0].plot(x_coarse, u1_test_coarse[sample_index], marker="s", c=colors["orange"], label="True $u(t=1, x)$")
    axes[0].plot(x_coarse, model_output_coarse, c=colors["red"], ls="--", marker="s", label="Predicted $u(t=0, x)$")
    axes[0].set(
        xlabel="Space $x$",
        xlim=xlim,
        ylabel="Solution $u$",
        title="Coarse resolution (used during training)",
    )

    axes[1].plot(x_fine, u0_test_fine[sample_index, :, 0], c="black", label="Initial condition $u(t=0, x)$")
    axes[1].plot(x_fine, u1_test_fine[sample_index], c=colors["orange"], label="True $u(t=1, x)$")
    axes[1].plot(x_fine, model_output_fine, c=colors["red"], ls="--", label="Predicted $u(t=0, x)$")
    axes[1].set(
        xlabel="Space $x$",
        xlim=xlim,
        title="Fine resolution",
    )
    # Combine handles and labels from both axes
    handles, labels = axes[1].get_legend_handles_labels()

    # Place the legend below both plots, centered, in one line
    fig.legend(
        handles,
        labels,
        loc="lower center",
        bbox_to_anchor=(0.5, -0.08),
        ncol=len(labels),
        frameon=True,
    )
    fig.suptitle(f"Zero-shot super-resolution (Test case {sample_index})", fontsize=20)
    plt.tight_layout()
    plt.show()

interactive(children=(BoundedIntText(value=0, description='Sample Index', max=204), Output()), _dom_classes=('…

# Task B: Halved domain

You might have noticed the checkbox `Half domain (Only in the second task!)` in section **2. Data subsampling and enrichment**. Now its time to activate it. It will use only the first half of our training data (halfing in the spacial domain).

Rerun the training and evaluation with the halfed data. What do you observe and how can your observations be explained?